# Frozen TESS scientific grid

This notebook inspects the pre-registered version-one grid without launching the production-scale simulations. The full run is intentionally an explicit action because the CVZ-like held-out cell and thousands of calibration realisations are expensive. EACF maps use zero-padded FFT autocorrelations, but the production experiment remains suitable for a compute node rather than a notebook session.

In [ ]:
from urdr import make_tess_scientific_grid

In [ ]:
grid = make_tess_scientific_grid()
manifest = grid.validation_plan.to_manifest()
manifest['name'], manifest['fingerprint'], len(manifest['cases'])

In [ ]:
cadence_counts = {
    case.name: case.window.time.size
    for case in grid.validation_plan.cases
}
cadence_counts

In [ ]:
case_table = [
    {
        'case': row.case,
        'split': row.split,
        'regime': row.regime,
        'signal': row.signal_level,
        'window': row.window_class,
    }
    for row in grid.metadata
]
case_table

In [ ]:
[(candidate.a, candidate.b) for candidate in grid.background_candidates]

Run the production validation with checkpointing. Start in a new output directory: the corrected full-lag manifest has a different fingerprint from the provisional PR #10 grid.

```python
from urdr import run_synthetic_experiment

run = run_synthetic_experiment(
    grid.validation_plan,
    'results/tess-synthetic-v1',
    workers=8,
    resume=True,
)
```

The empirical-background arm uses `benchmark_empirical_grid(grid.background_cases, grid.background_candidates, ...)`. Select candidates on the training cells only, then call `assess_background_calibration` to check the held-out and regime-specific Pareto frontiers. Use `assess_validation` on the synthetic-validation tables to keep detection performance and probability reliability separate.